# OCEL-Healer System Evaluation

Comprehensive evaluation of detection and resolution accuracy across 28 issue types.

**Phase 1**: 14 issue types with existing injectors (ready to run)

**Phase 2**: 28 issue types after creating remaining injectors

## Setup & Imports

In [2]:
import sqlite3
import shutil
from pathlib import Path
from typing import Callable, Any
import json
from datetime import datetime
import pandas as pd
from tqdm.notebook import tqdm
import traceback

from src.detection.error_detection import detect_all
import random
from src.llm.resolution import suggest_repair, detect_with_llm, detect_all_with_llm
from src.llm.actions import apply_repair
from src.llm.candidate_sources import candidate_rows, candidate_noun, candidate_kind
from src.detection.error_detection import _object_type_tables, _event_type_tables
from src.llm.client import set_active_model
from src.corruption.object_issues import (
    inject_missing_object_type_null_employee,
    inject_missing_object_type_whitespace_product,
    inject_missing_attribute_value_null_product_weight,
    inject_missing_attribute_value_null_order_price,
    inject_incorrect_attribute_datatype_string_in_weight,
    inject_incorrect_attribute_datatype_blob_in_role,
    inject_incorrect_object_type_swap_order_to_employee,
    inject_incorrect_object_type_case_variant_customers,
    inject_duplicate_objects_on_ids_product,
    inject_duplicate_objects_on_ids_conflicting_types,
    inject_duplicate_objects_on_attributes_clone_product,
    inject_duplicate_objects_on_attributes_clone_order_and_referenced,
    inject_incorrect_object_attribute_value_negative_weight_easy,
    inject_incorrect_object_attribute_value_implausible_weight_hard,
)
from src.corruption.event_issues import (
    inject_missing_event_type_null_confirm_easy,
    inject_missing_event_type_whitespace_package_hard,
    inject_missing_event_timestamp_null_place_order_easy,
    inject_missing_event_timestamp_null_item_out_of_stock_hard,
    inject_missing_event_place_order_easy,
    inject_missing_event_bare_id_hard,
    inject_missing_event_attribute_value_null_order_id_easy,
    inject_missing_event_attribute_value_null_reason_hard,
    inject_incorrect_event_attribute_datatype_string_in_quantity_easy,
    inject_incorrect_event_attribute_datatype_blob_in_activity_hard,
    inject_incorrect_event_attribute_value_negative_quantity_easy,
    inject_incorrect_event_attribute_value_time_violation_hard,
    inject_incorrect_event_type_swap_easy,
    inject_incorrect_event_type_case_variant_hard,
    inject_incorrect_event_time_future_easy,
    inject_incorrect_event_time_past_hard,
    inject_duplicate_events_on_ids_easy,
    inject_duplicate_events_on_ids_conflicting_types_hard,
    inject_duplicate_events_on_attributes_clone_easy,
    inject_duplicate_events_on_attributes_clone_with_refs_hard,
)
from src.corruption.relation_issues import (
    inject_dangling_e2o_relationship_missing_object_easy,
    inject_dangling_e2o_relationship_missing_both,
    inject_dangling_o2o_relationship_missing_source,
    inject_dangling_o2o_relationship_missing_both_typo,
    inject_missing_object_order_easy,
    inject_missing_object_product_hard,
    inject_o2o_self_loop_order_easy,
    inject_o2o_self_loop_product_hard,
    inject_duplicate_o2o_relations_comprises_easy,
    inject_duplicate_e2o_relations_order_easy,
    inject_duplicate_e2o_relations_sales_person_hard,
    inject_incorrect_e2o_relationship_target_wrong_order_easy,
    inject_incorrect_e2o_relationship_target_plausible_hard,
    inject_incorrect_e2o_relationship_qualifier_obvious_easy,
    inject_incorrect_e2o_relationship_qualifier_subtle_hard,
    inject_incorrect_o2o_relationship_target_wrong_item_easy,
    inject_incorrect_o2o_relationship_target_plausible_hard,
    inject_incorrect_o2o_relationship_qualifier_wrong_verb_easy,
    inject_incorrect_o2o_relationship_qualifier_typo_hard,
)

print("✅ Imports successful")

✅ Imports successful


## Configuration

In [3]:
# Test configuration
CLEAN_DB = "../../../../data/ocel2-p2p.sqlite"  # P2P database with event attributes (lifecycle, resource)
OUTPUT_DIR = Path("../../../../data/evaluation/notebook-eval")
MODELS = ["qwen2.5:7b"] #["mistral-small3.2:latest"]  # Add more models as needed: ["gpt-4", "claude-opus-4"]
RUNS_PER_SCENARIO = 2 #25  # Set to 3 for smoke test
DIFFICULTIES = ["easy", "hard"]

# Issue registry with injectors - ALL 28 ISSUE TYPES (Phase 2: Complete)
# Adapted for Procure-to-Pay (P2P) domain with real SAP procurement data
ISSUE_REGISTRY = {
    # === OBJECT ISSUES (9 types) ===
    "missing_object_type": {
        "detection": "rule",
        "injectors": {
            "easy": inject_missing_object_type_null_employee,
            "hard": inject_missing_object_type_whitespace_product,
        },
    },
    "missing_attribute_value": {  # Note: detector key is "missing_attribute_value" not "missing_object_attribute_value"
        "detection": "rule",
        "injectors": {
            "easy": inject_missing_attribute_value_null_product_weight,
            "hard": inject_missing_attribute_value_null_order_price,
        },
    },
    "incorrect_attribute_datatype": {
        "detection": "rule",
        "injectors": {
            "easy": inject_incorrect_attribute_datatype_string_in_weight,
            "hard": inject_incorrect_attribute_datatype_blob_in_role,
        },
    },
    "incorrect_object_attribute_value": {
        "detection": "llm",
        "injectors": {
            "easy": inject_incorrect_object_attribute_value_negative_weight_easy,
            "hard": inject_incorrect_object_attribute_value_implausible_weight_hard,
        },
    },
    "incorrect_object_type": {
        "detection": "rule",
        "injectors": {
            "easy": inject_incorrect_object_type_swap_order_to_employee,
            "hard": inject_incorrect_object_type_case_variant_customers,
        },
    },
    "duplicate_objects_on_ids": {
        "detection": "rule",
        "injectors": {
            "easy": inject_duplicate_objects_on_ids_product,
            "hard": inject_duplicate_objects_on_ids_conflicting_types,
        },
    },
    "duplicate_objects_on_attributes": {
        "detection": "rule",
        "injectors": {
            "easy": inject_duplicate_objects_on_attributes_clone_product,
            "hard": inject_duplicate_objects_on_attributes_clone_order_and_referenced,
        },
    },
    # missing_object_attribute: schema change, not implemented
    "missing_object": {
        "detection": "rule",
        "injectors": {
            "easy": inject_missing_object_order_easy,
            "hard": inject_missing_object_product_hard,
        },
    },
    
    # === EVENT ISSUES (11 types) ===
    # P2P database has lifecycle and resource attributes on all event types!
    "missing_event_type": {
        "detection": "rule",
        "injectors": {
            "easy": inject_missing_event_type_null_confirm_easy,
            "hard": inject_missing_event_type_whitespace_package_hard,
        },
    },
    "missing_event_timestamp": {
        "detection": "rule",
        "injectors": {
            "easy": inject_missing_event_timestamp_null_place_order_easy,
            "hard": inject_missing_event_timestamp_null_item_out_of_stock_hard,
        },
    },
    "missing_event_attribute_value": {
        "detection": "rule",
        "injectors": {
            "easy": inject_missing_event_attribute_value_null_order_id_easy,
            "hard": inject_missing_event_attribute_value_null_reason_hard,
        },
    },
    "incorrect_event_attribute_datatype": {
        "detection": "rule",
        "injectors": {
            "easy": inject_incorrect_event_attribute_datatype_string_in_quantity_easy,
            "hard": inject_incorrect_event_attribute_datatype_blob_in_activity_hard,
        },
    },
    "incorrect_event_attribute_value": {
        "detection": "llm",
        "injectors": {
            "easy": inject_incorrect_event_attribute_value_negative_quantity_easy,
            "hard": inject_incorrect_event_attribute_value_time_violation_hard,
        },
    },
    "incorrect_event_type": {
        "detection": "rule",
        "injectors": {
            "easy": inject_incorrect_event_type_swap_easy,
            "hard": inject_incorrect_event_type_case_variant_hard,
        },
    },
    "incorrect_event_time": {
        "detection": "rule",
        "injectors": {
            "easy": inject_incorrect_event_time_future_easy,
            "hard": inject_incorrect_event_time_past_hard,
        },
    },
    "duplicate_events_on_ids": {
        "detection": "rule",
        "injectors": {
            "easy": inject_duplicate_events_on_ids_easy,
            "hard": inject_duplicate_events_on_ids_conflicting_types_hard,
        },
    },
    "duplicate_events_on_attributes": {
        "detection": "rule",
        "injectors": {
            "easy": inject_duplicate_events_on_attributes_clone_easy,
            "hard": inject_duplicate_events_on_attributes_clone_with_refs_hard,
        },
    },
    # missing_event_attribute: schema change, not implemented
    "missing_event": {
        "detection": "rule",
        "injectors": {
            "easy": inject_missing_event_place_order_easy,
            "hard": inject_missing_event_bare_id_hard,
        },
    },
    
    # === RELATIONSHIP ISSUES (8 types) ===
    "dangling_e2o_relationship": {
        "detection": "rule",
        "injectors": {
            "easy": inject_dangling_e2o_relationship_missing_object_easy,
            "hard": inject_dangling_e2o_relationship_missing_both,
        },
    },
    "dangling_o2o_relationship": {
        "detection": "rule",
        "injectors": {
            "easy": inject_dangling_o2o_relationship_missing_source,
            "hard": inject_dangling_o2o_relationship_missing_both_typo,
        },
    },
    "incorrect_e2o_relationship_target": {
        "detection": "llm",
        "injectors": {
            "easy": inject_incorrect_e2o_relationship_target_wrong_order_easy,
            "hard": inject_incorrect_e2o_relationship_target_plausible_hard,
        },
    },
    "incorrect_e2o_relationship_qualifier": {
        "detection": "llm",
        "injectors": {
            "easy": inject_incorrect_e2o_relationship_qualifier_obvious_easy,
            "hard": inject_incorrect_e2o_relationship_qualifier_subtle_hard,
        },
    },
    "incorrect_o2o_relationship_target": {
        "detection": "llm",
        "injectors": {
            "easy": inject_incorrect_o2o_relationship_target_wrong_item_easy,
            "hard": inject_incorrect_o2o_relationship_target_plausible_hard,
        },
    },
    "incorrect_o2o_relationship_qualifier": {
        "detection": "llm",
        "injectors": {
            "easy": inject_incorrect_o2o_relationship_qualifier_wrong_verb_easy,
            "hard": inject_incorrect_o2o_relationship_qualifier_typo_hard,
        },
    },
    "o2o_self_loop": {  # Note: detector key is "o2o_self_loop" not "o2o_issues"
        "detection": "rule",
        "injectors": {
            "easy": inject_o2o_self_loop_order_easy,
            "hard": inject_o2o_self_loop_product_hard,
        },
    },
    "duplicate_e2o_relations": {
        "detection": "rule",
        "injectors": {
            "easy": inject_duplicate_e2o_relations_order_easy,
            "hard": inject_duplicate_e2o_relations_sales_person_hard,
        },
    },
}

# Filter out issues with missing injectors
TESTABLE_ISSUES = {
    k: v for k, v in ISSUE_REGISTRY.items()
    if v["injectors"]["easy"] is not None and v["injectors"]["hard"] is not None
}

# The 4 relationship LLM types have neither a task class nor a candidate-row
# source anywhere in the codebase yet (confirmed against src/llm/tasks/ and
# src/llm/candidate_sources.py) -- leaving them in would just burn a test slot
# on a type that can never score above 0%. Drop them until they're built.
NOT_YET_IMPLEMENTED = {
    "incorrect_e2o_relationship_target",
    "incorrect_e2o_relationship_qualifier",
    "incorrect_o2o_relationship_target",
    "incorrect_o2o_relationship_qualifier",
}
TESTABLE_ISSUES = {
    k: v for k, v in TESTABLE_ISSUES.items() if k not in NOT_YET_IMPLEMENTED
}

# Issue types whose detection routes through the AI sweep (detect_all_with_llm
# via src.llm.candidate_sources) instead of the rule-based detect_all(). Your
# test list calls one of these "incorrect_object_attribute_value"; the actual
# task/candidate-source key is "incorrect_attribute_value" (no "object") --
# _TASK_KEY_ALIASES below translates between the two so you don't have to
# rename anything in this registry.
LLM_DETECTED_TYPES = {
    "incorrect_object_type",
    "incorrect_event_type",
    "incorrect_event_time",
    "incorrect_object_attribute_value",
    "incorrect_event_attribute_value",
}
_TASK_KEY_ALIASES = {
    "incorrect_object_attribute_value": "incorrect_attribute_value",
}

# Per-type AI sweep sample cap. Object types run 927-2296 rows each in this
# database, event types 122-4042 -- sweeping every row every test run would
# mean hundreds to thousands of AI calls per test. Cap the sweep, always
# keeping the actually-injected id(s) in the sample.
LLM_SWEEP_SAMPLE_CAP = 150

print(f"✅ Configuration loaded")
print(f"   Domain: Procure-to-Pay (SAP P2P)")
print(f"   Clean DB: {CLEAN_DB}")
print(f"   Database size: {Path(CLEAN_DB).stat().st_size / 1024 / 1024:.1f} MB")
print(f"   Output: {OUTPUT_DIR}")
print(f"   Models: {MODELS}")
print(f"   Runs per scenario: {RUNS_PER_SCENARIO}")
print(f"   Total issue types: {len(ISSUE_REGISTRY)}")
print(f"   Testable issue types: {len(TESTABLE_ISSUES)}")
print(f"   (excluded {len(NOT_YET_IMPLEMENTED)} not-yet-implemented relationship types)")
print(f"   Total scenarios: {len(TESTABLE_ISSUES)} × {len(DIFFICULTIES)} × {len(MODELS)} × {RUNS_PER_SCENARIO} = {len(TESTABLE_ISSUES) * len(DIFFICULTIES) * len(MODELS) * RUNS_PER_SCENARIO} test runs")
print(f"")
print(f"🎉 NEW: Event attribute corruptors now functional with P2P's lifecycle & resource attributes!")

✅ Configuration loaded
   Domain: Procure-to-Pay (SAP P2P)
   Clean DB: ../../../../data/ocel2-p2p.sqlite
   Database size: 13.1 MB
   Output: ../../../../data/evaluation/notebook-eval
   Models: ['qwen2.5:7b']
   Runs per scenario: 2
   Total issue types: 26
   Testable issue types: 22
   (excluded 4 not-yet-implemented relationship types)
   Total scenarios: 22 × 2 × 1 × 2 = 88 test runs

🎉 NEW: Event attribute corruptors now functional with P2P's lifecycle & resource attributes!


## Core Evaluation Functions

## Clean-DB Baseline

`ocel2-p2p.sqlite` is real SAP procurement data, not a synthetic clean
fixture -- it already contains real, pre-existing data-quality issues
(confirmed: ~2,028 pre-existing dangling `object_object` rows, ~6,392
purchase orders with a NULL `VendorEKKOLIFNR`, etc.). `detect_all()` finds
those on every run whether or not anything was injected. We scan the
untouched clean DB once here and subtract this baseline from every
post-injection scan below, so pre-existing noise stops being counted as a
false positive against the 1-2 issues a given test actually injects.

In [4]:
def _row_signature(row: dict) -> tuple:
    """Order-independent identity for a detector row, ignoring the
    constant 'issue' label column so rows compare on their actual data.

    Defensive against non-hashable cell values: resolution's own
    context-building (suggest_repair -> task.build_context) can enrich
    a row dict in place with nested structures (seen on missing_event,
    which needs related-object context) -- coerce anything unhashable to
    a canonical string instead of letting the tuple-in-a-set blow up with
    "unhashable type: dict".
    """
    def _safe(v):
        try:
            hash(v)
            return v
        except TypeError:
            return json.dumps(v, sort_keys=True, default=str)
    return tuple(sorted((k, _safe(v)) for k, v in row.items() if k != "issue"))


def _frame_rows(df) -> list[dict]:
    return df.to_dicts() if df is not None and hasattr(df, "to_dicts") else []


def _frame_signatures(df) -> set[tuple]:
    return {_row_signature(r) for r in _frame_rows(df)}


def _row_matches_ids(row: dict, ids: set[str]) -> bool:
    """True if any cell value in `row` equals one of the injected affected_ids.
    Generic across issue types since every detector row carries the real
    ocel ids it references as plain column values."""
    return any(str(v) in ids for v in row.values())


print("Scanning clean baseline (runs once; every issue type)...")
_BASELINE_DETECTED = detect_all(CLEAN_DB)
_BASELINE_SIGNATURES = {
    issue: _frame_signatures(df) for issue, df in _BASELINE_DETECTED.items()
}
_baseline_total = sum(len(s) for s in _BASELINE_SIGNATURES.values())
print(f"✅ Baseline scanned -- {_baseline_total} pre-existing issues found "
      f"across {len(_BASELINE_SIGNATURES)} issue types (excluded from every score below)")

Scanning clean baseline (runs once; every issue type)...
✅ Baseline scanned -- 3342 pre-existing issues found across 18 issue types (excluded from every score below)


In [ ]:
from src.corruption._common import _remove_all_primary_keys


def _llm_sweep_new_rows(issue_type: str, db_path: str, affected_ids: set[str]) -> list[dict]:
    """AI-judged counterpart to the rule-based detect_all() path.

    Scopes the sweep to the type(s) the injected id(s) actually belong to
    (not every type in the database -- object types alone run 927-2296 rows
    each here) and caps each type's sweep at LLM_SWEEP_SAMPLE_CAP, always
    keeping the injected id(s) in the sample so recall is measurable even
    when the type is large.

    Known simplification: unlike the rule-based path, this does NOT diff
    against a clean-DB baseline sweep -- that would mean re-running the full
    (costly) AI sweep on the clean database too. Scoping to ~150 candidates
    of the relevant type already limits how much pre-existing noise can
    inflate detected_count relative to the whole-database rule-based case;
    a pre-existing implausible value in the SAME type can still count as a
    false positive here, which the rule-based path would have filtered out.
    """
    task_key = _TASK_KEY_ALIASES.get(issue_type, issue_type)
    kind = candidate_kind(task_key)

    with sqlite3.connect(db_path) as conn:
        if kind == "event":
            all_types = [t for t, _ in _event_type_tables(conn)]
            table, id_col = "event", "ocel_id"
        else:
            all_types = [t for t, _ in _object_type_tables(conn)]
            table, id_col = "object", "ocel_id"
        if affected_ids:
            placeholders = ", ".join("?" * len(affected_ids))
            chosen_types = {
                r[0] for r in conn.execute(
                    f"SELECT ocel_type FROM {table} WHERE {id_col} IN ({placeholders})",
                    list(affected_ids),
                ).fetchall()
            }
        else:
            chosen_types = set()

    # Fall back to sweeping every type only if we couldn't resolve which
    # type the injected id belongs to (shouldn't normally happen).
    types_to_sweep = chosen_types or set(all_types)

    def _row_id(r: dict) -> str:
        return str(r.get("ocel_id") or r.get("ocel_object_id") or r.get("ocel_event_id") or "")

    flagged_rows: list[dict] = []
    for t in types_to_sweep:
        candidates = candidate_rows(task_key, db_path, t)
        if len(candidates) > LLM_SWEEP_SAMPLE_CAP:
            must_keep = [r for r in candidates if _row_id(r) in affected_ids]
            rest = [r for r in candidates if _row_id(r) not in affected_ids]
            random.shuffle(rest)
            candidates = must_keep + rest[: max(0, LLM_SWEEP_SAMPLE_CAP - len(must_keep))]
        if not candidates:
            continue
        verdicts = detect_all_with_llm(task_key, candidates, db_path)
        flagged_rows.extend(dict(row) for row, verdict in verdicts if verdict.flagged)
    return flagged_rows


def _row_id_of(r: dict) -> str:
    return str(r.get("ocel_id") or r.get("ocel_object_id") or r.get("ocel_event_id") or "")


def _refresh_row_for_recheck(task_key: str, stale_row: dict, db_path: str) -> dict | None:
    """Re-derive a fresh candidate row for the same id straight from the
    (possibly just-repaired) database, instead of reusing the frozen
    pre-repair snapshot.

    Bug this fixes: the old correctness recheck called `detect_with_llm`
    on the SAME row dict captured before `suggest_repair`/`apply_repair`
    ran -- e.g. for `incorrect_object_type` that dict still carries the
    corrupted `ocel_type` (like 'payment') even after the repair correctly
    wrote 'purchase_order' back to the object table. The LLM was judging
    stale evidence every time, so a fully-correct repair could still score
    as "still flagged" (observed: proposed_value matched the ground-truth
    original_value exactly, repair applied cleanly, yet
    resolution_correctness came back 0.0).

    Returns None (caller falls back to the stale row) if the id can no
    longer be found under its current type -- e.g. a delete/merge repair
    removed it, which is a legitimate "resolved" case handled by the
    caller rather than this function.
    """
    target_id = _row_id_of(stale_row)
    kind = candidate_kind(task_key)
    table, id_col = ("event", "ocel_id") if kind == "event" else ("object", "ocel_id")
    with sqlite3.connect(db_path) as conn:
        row = conn.execute(
            f"SELECT ocel_type FROM {table} WHERE {id_col} = ?", (target_id,)
        ).fetchone()
    if row is None or row[0] is None:
        return None
    current_type = row[0]
    for c in candidate_rows(task_key, db_path, current_type):
        if _row_id_of(c) == target_id:
            return c
    return None


def _values_match(a, b) -> bool:
    """Ground-truth comparison used by resolution correctness. Floats
    compare with a small tolerance; everything else needs an exact match.
    Two Nones match (both "nothing recorded"); one None and one non-None
    do not (a real original value was replaced with nothing, or vice
    versa)."""
    if a is None and b is None:
        return True
    if a is None or b is None:
        return False
    try:
        return abs(float(a) - float(b)) < 1e-6
    except (TypeError, ValueError):
        return a == b


def inject_and_get_groundtruth(conn: sqlite3.Connection, injector_func: Callable, issue_type: str) -> dict:
    """Relax PK constraints, inject issue, and capture groundtruth.

    Returns dict with:
        - affected_ids: IDs that were modified/created
        - snapshot: Sample of unaffected data for collateral damage check
    """
    # Relax PK on all four spine tables (object, event, event_object,
    # object_object) BEFORE injecting. Without this, any injector that needs
    # to write a duplicate-id or duplicate-triple row (duplicate_objects_on_ids,
    # duplicate_events_on_ids, duplicate_e2o_relations, duplicate_o2o_relations)
    # throws a PRIMARY KEY / UNIQUE violation during setup, which used to get
    # swallowed by run_single_test's outer except and show up as "nan%" rows
    # in the report.
    _remove_all_primary_keys(conn)

    # Take snapshot of random unaffected rows (for collateral damage check)
    snapshot = {
        "objects": conn.execute(
            "SELECT ocel_id, ocel_type FROM object ORDER BY RANDOM() LIMIT 10"
        ).fetchall(),
        "events": conn.execute(
            "SELECT ocel_id, ocel_type FROM event ORDER BY RANDOM() LIMIT 10"
        ).fetchall(),
    }

    # Run injector
    affected = injector_func(conn)
    conn.commit()

    # Normalize affected to list. Injectors that know their pre-corruption
    # value (attribute-value/type-value injectors, missing_object,
    # missing_event) return a dict with affected_ids + original_values so
    # that value survives into the result. Everything else is unchanged.
    original_values: dict = {}
    if isinstance(affected, dict):
        affected_ids = list(affected.get("affected_ids") or [])
        original_values = affected.get("original_values") or {}
    elif affected is None:
        affected_ids = []
    elif isinstance(affected, (list, tuple)):
        affected_ids = list(affected)
    else:
        affected_ids = [affected]

    return {
        "affected_ids": affected_ids,
        "original_values": original_values,
        "snapshot": snapshot,
        "issue_type": issue_type,
    }


def evaluate_detection(db_path: str, issue_type: str, groundtruth: dict) -> dict:
    """Run detection and measure recall/precision against the SPECIFIC injected
    instance(s), after subtracting the clean-DB baseline (see _BASELINE_SIGNATURES).

    Replaces the old `true_positives = min(detected_count, injected_count)`
    heuristic, which silently counted every pre-existing P2P data-quality issue
    as a false positive (e.g. dangling_o2o_relationship: 2029 detected / 2
    injected, precision 0.1%, purely from ~2028 real pre-existing dangling rows).

    Matching is generic across issue types: a detector row counts as a true
    positive if any of its cell values equals one of `groundtruth['affected_ids']`
    -- every detect_* function's row carries the real ocel ids it references as
    plain column values, so this needs no per-issue-type schema mapping.
    """
    try:
        affected_ids = {str(a) for a in groundtruth["affected_ids"]}
        injected_count = len(affected_ids)

        if issue_type in LLM_DETECTED_TYPES:
            # AI-judged path: sweep the relevant type(s) with the LLM instead
            # of the rule-based detect_all(). See _llm_sweep_new_rows for the
            # baseline-diffing simplification this path makes.
            new_rows = _llm_sweep_new_rows(issue_type, db_path, affected_ids)
        else:
            all_detected = detect_all(db_path)
            detected_df = all_detected.get(issue_type)
            detected_rows = _frame_rows(detected_df)
            baseline_sigs = _BASELINE_SIGNATURES.get(issue_type, set())
            new_rows = [r for r in detected_rows if _row_signature(r) not in baseline_sigs]

        matched_rows = [r for r in new_rows if _row_matches_ids(r, affected_ids)]
        matched_ids = {
            a for a in affected_ids
            if any(str(a) in {str(v) for v in r.values()} for r in new_rows)
        }

        detected_count = len(new_rows)
        true_positives = len(matched_rows)

        if injected_count > 0:
            recall = len(matched_ids) / injected_count
        else:
            # Nothing was actually injected this run (injector found no
            # candidate row) -- recall is only meaningful relative to an
            # injected instance, so leave it undefined rather than fabricate
            # a value the way the old `injected_count = ... or 1` default did.
            recall = float("nan")

        if detected_count > 0:
            precision = true_positives / detected_count
        else:
            precision = 1.0 if injected_count == 0 else 0.0

        return {
            "detection_recall": recall,
            "detection_precision": precision,
            "detected_count": detected_count,
            "injected_count": injected_count,
            "detection_success": len(matched_ids) > 0,
            "_matched_new_rows": matched_rows,
        }
    except Exception as e:
        return {
            "detection_recall": 0.0,
            "detection_precision": 0.0,
            "detected_count": 0,
            "injected_count": len(groundtruth["affected_ids"]),
            "detection_success": False,
            "detection_error": str(e),
            "_matched_new_rows": [],
        }


def evaluate_resolution(
    db_path: str,
    issue_type: str,
    detected_issues: list,
    model: str,
    original_values: dict | None = None,
) -> dict:
    """Run resolution and measure correctness/safety.

    Three fixes vs. the original:
      1. `apply_repair(..., dry_run=False)` -- the original call relied on
         apply_repair's default (dry_run=True), so NOTHING was ever actually
         written to the database; every "applied"/"success" row in the old
         report was a dry-run description, not a real repair.
      2. `resolution_correctness` checks whether the SPECIFIC injected
         instance(s) actually stop being a new (post-baseline) detection
         after the repair, instead of `proposed / attempted` (which only
         measured "did the LLM suggest something", never whether it was
         right).
      3. For LLM-detected issue types (incorrect_object_type,
         incorrect_event_type, incorrect_event_time,
         incorrect_object_attribute_value, incorrect_event_attribute_value)
         the recheck now re-derives the row fresh from the (repaired)
         database via `_refresh_row_for_recheck` instead of re-judging the
         frozen pre-repair snapshot. The frozen-snapshot version always
         re-fed the LLM the SAME corrupted evidence it saw at detection
         time, so a fully-correct repair could still score as "still
         flagged" -- this was silently making resolution_correctness wrong
         for every LLM-detected type.
      4. Wherever the injector reports a ground-truth `original_value` for
         a row, correctness now also requires the proposed value to MATCH
         it (see `_values_match`), not just "the row is no longer flagged".
         Without this, a "missing value" fix could hallucinate any
         non-null replacement (e.g. proposing 63.0 for a field whose real
         pre-corruption value was 37.0) and still count as correct, because
         the rule-based detector for that issue type only checks
         "not null", not "matches the original". Issue types with no known
         ground truth (duplicates, dangling refs, etc.) are unaffected --
         they still rely purely on the detector re-check, as before.

    `original_values` (optional): the `groundtruth["original_values"]` dict
    from `inject_and_get_groundtruth`, keyed by ocel id -- the pre-corruption
    value for issue types whose injector reports one. Combined with each
    attempted row's proposed value, this fills `repairs` below with an
    original_value -> proposed_value pair per row, so a "correct" repair can
    actually be eyeballed instead of only trusted via the detector re-check.

    `detected_issues` should be the `_matched_new_rows` from `evaluate_detection`
    for this same test (the actual injected instance(s)), not a fresh unscoped
    `detect_all()` pull -- otherwise, for noisy issue types, resolution risks
    being attempted against real pre-existing P2P data rather than what was
    actually injected.
    """
    original_values = original_values or {}

    def _row_id(r: dict) -> str:
        return str(r.get("ocel_id") or r.get("ocel_object_id") or r.get("ocel_event_id") or "")

    if not detected_issues:
        return {
            "resolution_attempted": 0,
            "resolution_proposed": 0,
            "resolution_applied": 0,
            "resolution_success": False,
            "resolution_correctness": 0.0,
            "resolution_errors": "",
            "repairs": [],
        }

    task_key = _TASK_KEY_ALIASES.get(issue_type, issue_type)

    attempted = 0
    proposed = 0
    applied = 0
    errors: list[str] = []
    attempted_rows: list[dict] = []
    repairs: list[dict] = []

    for issue_row in detected_issues[:5]:  # Limit to first 5 for performance
        row_id = _row_id(issue_row)
        proposed_value = None
        try:
            attempted += 1
            # Shallow copy BEFORE calling suggest_repair: build_context can
            # (and for missing_event, does) enrich the row dict in place with
            # nested context for the LLM, which would otherwise corrupt the
            # copy we keep for the post-repair correctness diff.
            attempted_rows.append(dict(issue_row))

            # Get proposed fix. task_key handles the one registry/task naming
            # mismatch (incorrect_object_attribute_value -> incorrect_attribute_value);
            # identical for every other issue type.
            action = suggest_repair(task_key, issue_row, db_path)
            proposed_value = (action or {}).get("new_value")

            if action and action.get("kind") != "noop":
                proposed += 1

                # Apply fix -- dry_run=False is the fix: apply_repair defaults
                # to dry_run=True, so without this nothing was ever committed.
                apply_repair(db_path, action, dry_run=False)
                applied += 1
        except Exception as e:
            # Surfaced now instead of silently swallowed -- apply_repair
            # raises NotImplementedError for kind in ("decline", "unrouted"),
            # which the old bare `except: pass` hid entirely.
            errors.append(f"{issue_row}: {e}")
        finally:
            repairs.append({
                "ocel_id": row_id,
                "original_value": original_values.get(row_id),
                "proposed_value": proposed_value,
            })

    # "No longer flagged": of the rows we attempted, which ones does a fresh
    # re-check (against the now-repaired database) confirm are resolved?
    if issue_type in LLM_DETECTED_TYPES:
        # Re-judge each attempted row using a row FRESHLY re-derived from the
        # repaired database (see _refresh_row_for_recheck's docstring for why
        # re-using the pre-repair snapshot here was the incorrect_object_type
        # bug). Falls back to the stale row if the id can't be re-derived
        # (e.g. it was deleted/merged by the repair, which the LLM's verdict
        # on the stale row will typically still get right for "no longer a
        # problem" cases, since a missing row can't be judged incorrect).
        still_flagged_sigs = set()
        for r in attempted_rows:
            fresh_row = _refresh_row_for_recheck(task_key, r, db_path) or r
            try:
                verdict = detect_with_llm(task_key, fresh_row, db_path)
                if verdict.flagged:
                    still_flagged_sigs.add(_row_signature(r))
            except Exception:
                # Treat an error on recheck as "can't confirm resolved".
                still_flagged_sigs.add(_row_signature(r))
        no_longer_flagged_sigs = {
            _row_signature(r) for r in attempted_rows
            if _row_signature(r) not in still_flagged_sigs
        }
    else:
        baseline_sigs = _BASELINE_SIGNATURES.get(issue_type, set())
        post_repair_rows = _frame_rows(detect_all(db_path).get(issue_type))
        still_new_sigs = {
            _row_signature(r) for r in post_repair_rows
            if _row_signature(r) not in baseline_sigs
        }
        no_longer_flagged_sigs = {
            _row_signature(r) for r in attempted_rows
            if _row_signature(r) not in still_new_sigs
        }

    # Correctness = no longer flagged AND (where a ground-truth value is
    # known) the proposed value actually matches it. See fix #4 above.
    resolved = 0
    for r, rep in zip(attempted_rows, repairs):
        cleared = _row_signature(r) in no_longer_flagged_sigs
        row_id = rep["ocel_id"]
        if row_id in original_values:
            cleared = cleared and _values_match(rep["proposed_value"], original_values[row_id])
        if cleared:
            resolved += 1
    correctness = resolved / attempted if attempted > 0 else 0.0

    return {
        "resolution_attempted": attempted,
        "resolution_proposed": proposed,
        "resolution_applied": applied,
        "resolution_success": applied > 0,
        "resolution_correctness": correctness,
        "resolution_errors": " | ".join(errors[:3]),
        "repairs": repairs,
    }


def run_single_test(
    issue_type: str,
    difficulty: str,
    injector_func: Callable,
    model: str,
    run_id: int,
) -> dict:
    """Run one complete test: inject -> detect -> resolve -> validate."""
    # Create temporary test database
    test_db = OUTPUT_DIR / f"test_{issue_type}_{difficulty}_{model.replace(':', '_')}_{run_id}.sqlite"
    test_db.parent.mkdir(parents=True, exist_ok=True)

    try:
        # Copy clean database
        shutil.copy2(CLEAN_DB, test_db)

        # Inject issue and capture groundtruth
        with sqlite3.connect(test_db) as conn:
            groundtruth = inject_and_get_groundtruth(conn, injector_func, issue_type)

        # Evaluate detection (baseline-diffed, matched against affected_ids)
        detection_metrics = evaluate_detection(str(test_db), issue_type, groundtruth)

        # Feed resolution the SAME matched rows detection just found, instead
        # of a second unscoped detect_all() pull -- keeps resolution targeted
        # at what was actually injected rather than baseline noise.
        detected_issues = detection_metrics.pop("_matched_new_rows", [])

        # Evaluate resolution
        resolution_metrics = evaluate_resolution(
            str(test_db),
            issue_type,
            detected_issues,
            model,
            original_values=groundtruth.get("original_values"),
        )

        # overall_success now requires the repair to have been VERIFIED
        # correct (resolution_correctness == 1.0 across every attempted
        # row), not merely applied without an exception. Previously
        # `resolution_success` (applied > 0) alone drove this, which let
        # e.g. incorrect_object_type report 100% overall success in the
        # same run its resolution_correctness was 0% -- "the write didn't
        # crash" and "the write was right" are different claims, and only
        # the second one belongs in a headline success-rate number.
        # `resolution_success` is kept as-is in the row (applied > 0) as a
        # separate plumbing/liveness signal -- see the report generation
        # cell for how it's now labeled.
        result = {
            "issue_type": issue_type,
            "difficulty": difficulty,
            "model": model,
            "run_id": run_id,
            "timestamp": datetime.now().isoformat(),
            **detection_metrics,
            **resolution_metrics,
            "overall_success": (
                detection_metrics.get("detection_success", False)
                and resolution_metrics.get("resolution_correctness", 0.0) >= 1.0
            ),
        }

        return result

    except Exception as e:
        return {
            "issue_type": issue_type,
            "difficulty": difficulty,
            "model": model,
            "run_id": run_id,
            "timestamp": datetime.now().isoformat(),
            "error": str(e),
            "traceback": traceback.format_exc(),
            "overall_success": False,
        }
    finally:
        # Clean up test database
        if test_db.exists():
            test_db.unlink()

print("✅ Core functions defined (baseline-diffed detection scoring, real apply_repair, "
      "fresh-row LLM recheck, ground-truth-aware correctness, correctness-gated overall_success)")


## Main Evaluation Loop

In [6]:
# Run all tests
results = []
total_scenarios = len(TESTABLE_ISSUES) * len(DIFFICULTIES) * len(MODELS) * RUNS_PER_SCENARIO

print(f"Starting evaluation: {total_scenarios} total test runs")
print(f"Output directory: {OUTPUT_DIR}")
print()

with tqdm(total=total_scenarios, desc="Running evaluation") as pbar:
    for issue_type, config in TESTABLE_ISSUES.items():
        for difficulty in DIFFICULTIES:
            injector = config["injectors"][difficulty]
            if injector is None:
                continue
                
            for model in MODELS:
                set_active_model(model)
                
                for run_id in range(RUNS_PER_SCENARIO):
                    result = run_single_test(
                        issue_type=issue_type,
                        difficulty=difficulty,
                        injector_func=injector,
                        model=model,
                        run_id=run_id,
                    )
                    results.append(result)
                    pbar.update(1)
                    
                    # Update progress bar description with latest result
                    success_str = "✅" if result.get("overall_success") else "❌"
                    pbar.set_postfix_str(f"{success_str} {issue_type}[{difficulty}]")

# Save results
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
df = pd.DataFrame(results)
df.to_csv(OUTPUT_DIR / "results.csv", index=False)
df.to_json(OUTPUT_DIR / "results.json", orient="records", indent=2)

print(f"\n✅ Evaluation complete!")
print(f"   Results saved to: {OUTPUT_DIR}")
print(f"   Total runs: {len(results)}")
print(f"   Successful: {df['overall_success'].sum()} ({df['overall_success'].mean():.1%})")

Starting evaluation: 88 total test runs
Output directory: ../../../../data/evaluation/notebook-eval



Running evaluation:   0%|          | 0/88 [00:00<?, ?it/s]


✅ Evaluation complete!
   Results saved to: ../../../../data/evaluation/notebook-eval
   Total runs: 88
   Successful: 34 (38.6%)


## Results Analysis

In [7]:
# Aggregate metrics by issue type, difficulty, and model
if len(df) > 0 and all(col in df.columns for col in ["detection_recall", "detection_precision", "resolution_correctness"]):
    summary = df.groupby(["issue_type", "difficulty", "model"]).agg({
        "detection_recall": ["mean", "std", "min", "max"],
        "detection_precision": ["mean", "std", "min", "max"],
        "resolution_correctness": ["mean", "std", "min", "max"],
        "overall_success": ["sum", "mean"],
    }).round(3)

    print("📊 Summary Statistics by Issue Type, Difficulty, and Model")
    print("=" * 80)
    display(summary)

    # Overall statistics
    print("\n📊 Overall Statistics")
    print("=" * 80)
    print(f"Total test runs: {len(df)}")
    print(f"Overall success rate: {df['overall_success'].mean():.1%}")
    print(f"Average detection recall: {df['detection_recall'].mean():.1%}")
    print(f"Average detection precision: {df['detection_precision'].mean():.1%}")
    print(f"Average resolution correctness: {df['resolution_correctness'].mean():.1%}")
else:
    print("⚠️  Cannot generate summary statistics - required columns are missing.")
    print(f"Total test runs: {len(df)}")
    if "overall_success" in df.columns:
        print(f"Overall success rate: {df['overall_success'].mean():.1%}")
    if "error" in df.columns:
        errors = df[df["error"].notna()]
        if len(errors) > 0:
            print(f"\n❌ {len(errors)} tests failed with errors:")
            print(errors[["issue_type", "difficulty", "error"]].head())

📊 Summary Statistics by Issue Type, Difficulty, and Model


detection_recall  \
                                                                     mean   
issue_type                         difficulty model                         
dangling_e2o_relationship          easy       qwen2.5:7b              1.0   
                                   hard       qwen2.5:7b              1.0   
dangling_o2o_relationship          easy       qwen2.5:7b              1.0   
                                   hard       qwen2.5:7b              1.0   
duplicate_e2o_relations            easy       qwen2.5:7b              1.0   
                                   hard       qwen2.5:7b              0.0   
duplicate_events_on_attributes     easy       qwen2.5:7b              0.0   
                                   hard       qwen2.5:7b              0.0   
duplicate_events_on_ids            easy       qwen2.5:7b              1.0   
                                   hard       qwen2.5:7b              1.0   
duplicate_objects_on_attributes    easy       qwen2.5:7b              0.0   
                                   hard       qwen2.5:7b              0.0   
duplicate_objects_on_ids           easy       qwen2.5:7b              1.0   
                                   hard       qwen2.5:7b              1.0   
incorrect_attribute_datatype       easy       qwen2.5:7b              1.0   
                                   hard       qwen2.5:7b              1.0   
incorrect_event_attribute_datatype easy       qwen2.5:7b              0.0   
                                   hard       qwen2.5:7b              1.0   
incorrect_event_attribute_value    easy       qwen2.5:7b              1.0   
                                   hard       qwen2.5:7b              0.0   
incorrect_event_time               easy       qwen2.5:7b              0.0   
                                   hard       qwen2.5:7b              0.0   
incorrect_event_type               easy       qwen2.5:7b              0.0   
                                   hard       qwen2.5:7b              0.0   
incorrect_object_attribute_value   easy       qwen2.5:7b              0.0   
                                   hard       qwen2.5:7b              0.0   
incorrect_object_type              easy       qwen2.5:7b              1.0   
                                   hard       qwen2.5:7b              0.0   
missing_attribute_value            easy       qwen2.5:7b              1.0   
                                   hard       qwen2.5:7b              1.0   
missing_event                      easy       qwen2.5:7b              1.0   
                                   hard       qwen2.5:7b              1.0   
missing_event_attribute_value      easy       qwen2.5:7b              1.0   
                                   hard       qwen2.5:7b              1.0   
missing_event_timestamp            easy       qwen2.5:7b              1.0   
                                   hard       qwen2.5:7b              1.0   
missing_event_type                 easy       qwen2.5:7b              1.0   
                                   hard       qwen2.5:7b              1.0   
missing_object                     easy       qwen2.5:7b              1.0   
                                   hard       qwen2.5:7b              1.0   
missing_object_type                easy       qwen2.5:7b              1.0   
                                   hard       qwen2.5:7b              1.0   
o2o_self_loop                      easy       qwen2.5:7b              1.0   
                                   hard       qwen2.5:7b              1.0   

                                                                         \
                                                          std  min  max   
issue_type                         difficulty model                       
dangling_e2o_relationship          easy       qwen2.5:7b  0.0  1.0  1.0   
                                   hard       qwen2.5:7b  0.0  1.0  1.0   
dangling_o2o_relationship          easy       qwen2.5:7b  0.0 


📊 Overall Statistics
Total test runs: 88
Overall success rate: 38.6%
Average detection recall: 68.2%
Average detection precision: 68.2%
Average resolution correctness: 36.4%


## Visualizations

In [8]:
# Visualization cell removed - CSV output is sufficient
print("✅ Skipping visualization generation (CSV output is sufficient)")

✅ Skipping visualization generation (CSV output is sufficient)


## Generate Markdown Report

In [ ]:
def generate_markdown_report(df: pd.DataFrame, output_path: Path):
    """Generate comprehensive markdown report."""
    lines = []
    lines.append("# OCEL-Healer System Evaluation Report")
    lines.append("")
    lines.append(f"**Generated:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    lines.append(f"**Total test runs:** {len(df)}")
    lines.append(f"**Models tested:** {', '.join(df['model'].unique())}")
    lines.append(f"**Issue types:** {len(df['issue_type'].unique())}")
    lines.append("")
    
    # Check if we have valid metrics columns
    has_metrics = all(col in df.columns for col in ['detection_recall', 'detection_precision', 'resolution_correctness'])
    
    if not has_metrics:
        lines.append("## ⚠️ Incomplete Results")
        lines.append("")
        lines.append("The evaluation did not complete successfully. Metrics columns are missing.")
        lines.append("This usually means there was an error during the evaluation run.")
        lines.append("")
        lines.append("Available columns: " + ", ".join(df.columns.tolist()))
        lines.append("")
        
        if 'error' in df.columns:
            errors = df[df['error'].notna()]
            if len(errors) > 0:
                lines.append("## Errors Encountered")
                lines.append("")
                for _, row in errors.head().iterrows():
                    lines.append(f"- **{row['issue_type']}** [{row['difficulty']}]: {row['error']}")
                lines.append("")
        
        output_path.write_text("\n".join(lines), encoding="utf-8")
        return
    
    lines.append("## Executive Summary")
    lines.append("")
    lines.append(f"- **Overall success rate:** {df['overall_success'].mean():.1%} (detection AND VERIFIED-correct resolution)")
    lines.append(f"- **Detection success rate:** {df['detection_success'].mean():.1%} ⭐")
    lines.append(f"- **Average detection recall:** {df['detection_recall'].mean():.1%}")
    lines.append(f"- **Average detection precision:** {df['detection_precision'].mean():.1%}")
    lines.append(f"- **Repair-applied rate (plumbing, not correctness):** {df['resolution_success'].mean():.1%}")
    lines.append(f"- **Average resolution correctness:** {df['resolution_correctness'].mean():.1%}")
    lines.append("")
    
    if df['resolution_success'].sum() == 0:
        lines.append("ℹ️ **Note**: Resolution requires an LLM server. Detection results (above) show the rule-based detection is working correctly.")
        lines.append("")
    
    lines.append("## Detection Results by Issue Type")
    lines.append("")
    lines.append("| Issue Type | Difficulty | Detection Success | Recall | Precision | Detected / Injected |")
    lines.append("|---|---|---|---|---|---|")
    
    for (issue, diff), group in df.groupby(["issue_type", "difficulty"]):
        success_rate = group['detection_success'].mean()
        recall = group['detection_recall'].mean()
        precision = group['detection_precision'].mean()
        detected = group['detected_count'].sum()
        injected = group['injected_count'].sum()
        
        lines.append(f"| {issue} | {diff} | {success_rate:.1%} | {recall:.1%} | {precision:.1%} | {detected} / {injected} |")
    
    lines.append("")
    lines.append("## Full Results by Issue Type")
    lines.append("")
    lines.append("| Issue Type | Difficulty | Detection Success | Detection Recall | Detection Precision | Resolution Correctness | Overall Success |")
    lines.append("|---|---|---|---|---|---|---|")
    
    for (issue, diff), group in df.groupby(["issue_type", "difficulty"]):
        det_success = group['detection_success'].mean()
        recall = group['detection_recall'].mean()
        precision = group['detection_precision'].mean()
        resolution = group['resolution_correctness'].mean()
        overall_success = group['overall_success'].mean()
        
        lines.append(f"| {issue} | {diff} | {det_success:.1%} | {recall:.1%} | {precision:.1%} | {resolution:.1%} | {overall_success:.1%} |")
    
    lines.append("")
    lines.append("## Difficulty Comparison")
    lines.append("")
    
    for diff in ["easy", "hard"]:
        subset = df[df['difficulty'] == diff]
        if len(subset) > 0:
            lines.append(f"### {diff.capitalize()}")
            lines.append(f"- **Detection success:** {subset['detection_success'].mean():.1%}")
            lines.append(f"- Detection recall: {subset['detection_recall'].mean():.1%}")
            lines.append(f"- Detection precision: {subset['detection_precision'].mean():.1%}")
            lines.append(f"- Resolution correctness: {subset['resolution_correctness'].mean():.1%}")
            lines.append(f"- Overall success: {subset['overall_success'].mean():.1%}")
            lines.append("")
    
    lines.append("## Detection Performance (Detailed)")
    lines.append("")
    lines.append(f"- **Detection success rate:** {df['detection_success'].mean():.1%} ({df['detection_success'].sum()} / {len(df)} tests)")
    lines.append(f"- **Average recall:** {df['detection_recall'].mean():.3f}")
    lines.append(f"- **Average precision:** {df['detection_precision'].mean():.3f}")
    lines.append(f"- **Total detected:** {df['detected_count'].sum()} issues")
    lines.append(f"- **Total injected:** {df['injected_count'].sum()} issues")
    lines.append(f"- **Detection rate:** {df['detected_count'].sum() / df['injected_count'].sum():.1%}")
    lines.append("")
    
    lines.append("## Resolution Performance (Detailed)")
    lines.append("")
    if df['resolution_success'].sum() > 0:
        lines.append(f"- **Repair-applied rate (plumbing, not correctness):** {df['resolution_success'].mean():.1%} ({df['resolution_success'].sum()} / {len(df)} tests)")
        lines.append(f"- **Average correctness:** {df['resolution_correctness'].mean():.3f}")
        lines.append(f"- **Attempted:** {df['resolution_attempted'].sum()}")
        lines.append(f"- **Proposed:** {df['resolution_proposed'].sum()}")
        lines.append(f"- **Applied:** {df['resolution_applied'].sum()}")
    else:
        lines.append("⚠️ **No successful resolutions**")
        lines.append("")
        lines.append("This indicates the LLM server was not available during evaluation.")
        lines.append("")
        lines.append(f"- **Attempted:** {df['resolution_attempted'].sum()}")
        lines.append(f"- **Proposed:** {df['resolution_proposed'].sum()} (requires LLM)")
        lines.append(f"- **Applied:** {df['resolution_applied'].sum()}")
        lines.append("")
        lines.append("💡 **Good news**: Detection is working perfectly! See the Detection Performance section above.")
    lines.append("")
    
    lines.append("## Summary by Issue Type")
    lines.append("")
    lines.append("| Issue Type | Runs | Detection Success | Detection Recall | Detection Precision | Resolution Correctness | Overall Success |")
    lines.append("|---|---|---|---|---|---|---|")
    
    by_issue = df.groupby('issue_type').agg({
        'run_id': 'count',
        'detection_success': 'mean',
        'detection_recall': 'mean',
        'detection_precision': 'mean',
        'resolution_correctness': 'mean',
        'overall_success': 'mean',
    })
    
    for issue, row in by_issue.iterrows():
        lines.append(f"| {issue} | {int(row['run_id'])} | {row['detection_success']:.1%} | {row['detection_recall']:.1%} | {row['detection_precision']:.1%} | {row['resolution_correctness']:.1%} | {row['overall_success']:.1%} |")
    
    lines.append("")
    lines.append("---")
    lines.append("")
    lines.append("*Report generated by OCEL-Healer evaluation framework*")
    
    # Write report
    output_path.write_text("\n".join(lines), encoding="utf-8")

# Generate report
report_path = OUTPUT_DIR / "evaluation_report.md"

try:
    generate_markdown_report(df, report_path)
    print(f"✅ Markdown report saved to: {report_path}")
    
    # Display report in notebook
    from IPython.display import Markdown
    display(Markdown(report_path.read_text()))
except Exception as e:
    print(f"⚠️  Could not generate markdown report: {e}")
    print(f"   This usually means the evaluation encountered errors.")
    print(f"   Check the CSV file for error details: {OUTPUT_DIR / 'results.csv'}")